# Packages

In [ ]:
!pip install diffusers transformers accelerate --quiet
!pip install datasets --quiet
!pip install ftfy regex tqdm --quiet
!pip install git+https://github.com/openai/CLIP.git --quiet

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import os
import csv
import json
import time
import clip
import shutil
import random
from tqdm import tqdm
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn as nn
from torchvision import transforms
from torch.optim import AdamW
from torchvision.transforms.functional import to_pil_image
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
# from google.colab import files

# Loading and Preprocessing Data

In [ ]:
start = time.time()

!mkdir -p coco_dataset/images coco_dataset/annotations

!wget http://images.cocodataset.org/zips/train2017.zip -P coco_dataset/
!unzip -q coco_dataset/train2017.zip -d coco_dataset/images/

!wget http://images.cocodataset.org/zips/val2017.zip -P coco_dataset/
!unzip -q coco_dataset/val2017.zip -d coco_dataset/images/

!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P coco_dataset/
!unzip -q coco_dataset/annotations_trainval2017.zip -d coco_dataset/annotations/

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")



In [ ]:
def prepare_coco_caption_file(annotation_path, output_json, max_captions=None):
    """ Extracts image-caption pairs from COCO dataset and save them to simplified JSON file

    This function reads COCO annotations JSON file, extracts the image file names and associated
    captions, and saves them as a list of dictionaries to a new JSON file. Each entry contains a
    filename and corresponding caption.

    Args:
        annotation_path (str): Path to COCO annotations JSON file
        output_json (str): Path to save the cleaned COCO image-caption pairs JSON file
        max_captions (int, optional): Maximum no.of image-caption pairs to extract
    """

    with open(annotation_path, "r", encoding="utf-8") as f:
        annotations = json.load(f)

    image_id_to_file = {img["id"]: img["file_name"] for img in annotations["images"]}
    pairs = []

    for ann in annotations["annotations"]:
        image_id = ann["image_id"]
        caption = ann["caption"]
        file_name = image_id_to_file.get(image_id)
        if file_name:
            pairs.append({
                "file_name": file_name,
                "caption": caption
            })
        if max_captions and len(pairs) >= max_captions:
            break

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(pairs, f, indent=2)

    print(f"Saved {len(pairs)} image-caption pairs to {output_json}")


In [ ]:
def clean_caption_file(caption_file, image_folder, output_file):
    """ Cleans COCO image-caption pairs JSON file

    This function cleans COCO image-caption pair JSON file by removing the entries with missing
    image files, skips the corrupt image files, and ignore the empty captions.

    Args:
        caption_file (str): Path to the image-caption pair JSON file that needs to be cleaned
        image_folder (str): Path to the folder where COCO images are stored
        output_file (str): Path to save the verified COCO image-caption pairs JSON file
    """
    with open(caption_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    cleaned = []
    skipped = 0

    for item in tqdm(data, desc=f"Verifying {caption_file}"):
        image_path = os.path.join(image_folder, item["file_name"])

        if not os.path.exists(image_path):
            skipped += 1
            continue

        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception:
            skipped += 1
            continue

        if item["caption"].strip() == "":
            skipped += 1
            continue

        cleaned.append(item)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2)

    print(f"{len(cleaned)} valid image-caption pairs saved to '{output_file}'")
    print(f"{skipped} entries skipped due to missing/broken images or empty captions")


In [ ]:
def create_coco_subset(caption_file, output_file, num_images, captions_per_image):
    """ Create a subset of COCO dataset with specified image-caption pairs

    Reads a simplified COCO Caption file randomly selects a fixed number of images, and choose
    a specific no.of captions per image to build a subset.

    Args:
        caption_file (str): Path to input JSON file with image-caption pairs
        output_file (str): Path to save the output subset JSON file
        num_images (int): No.of unique images to include in the subset
        captions_per_image (int): No.of captions to be selected per image
    """

    with open(caption_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_to_captions = defaultdict(list)
    for entry in data:
        image_to_captions[entry["file_name"]].append(entry["caption"])

    all_images = list(image_to_captions.keys())
    random.shuffle(all_images)
    selected_images = all_images[:num_images]

    new_dataset = []
    for img in selected_images:
        captions = image_to_captions[img]
        chosen_captions = (
            random.sample(captions, captions_per_image)
            if len(captions) >= captions_per_image else captions
        )

        for caption in chosen_captions:
            new_dataset.append({"file_name": img, "caption": caption})

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(new_dataset, f, indent=2)

    print(f"New dataset created with {len(new_dataset)} samples")


### Train Data

In [ ]:
start = time.time()

prepare_coco_caption_file(
    annotation_path="coco_dataset/annotations/annotations/captions_train2017.json",
    output_json="coco_train.json",
    max_captions=None
)

clean_caption_file(
    caption_file="coco_train.json",
    image_folder="coco_dataset/images/train2017",
    output_file="coco_train_cleaned.json"
)


create_coco_subset(
    caption_file="coco_train_cleaned.json",
    output_file="coco_train_subset.json",
    num_images=100,
    captions_per_image=2
)

stop = time.time()
print(f"Time taken: {stop - start:.2f} seconds")


### Validation Data

In [ ]:
start = time.time()

prepare_coco_caption_file(
    annotation_path="coco_dataset/annotations/annotations/captions_val2017.json",
    output_json="coco_val.json",
    max_captions=None
)

clean_caption_file(
    caption_file="coco_val.json",
    image_folder="coco_dataset/images/val2017",
    output_file="coco_val_cleaned.json"
)

stop = time.time()

print(f"Time taken: {stop - start:.2f} seconds")

# Generate Text Embeddings

In [ ]:
def generate_clip_text_embeddings(
    caption_json_path,
    output_dir,
    batch_size=16,
    part_size=10000,
    model_name="openai/clip-vit-large-patch14"
):
    """Generated and saves CLIP embeddings for the image captions

    The function loads the image captions from the JSON file, uses the tokeninzer and
    CLIP text encoder to generate captions for each caption, and saves the results in
    parts to .pt file inside the given directory.

    Args:
        caption_json_path (str): Path to JSON file containing image-caption pairs
        output_dir (str): Directory to save the generated embedding files
        batch_size (int, optional): No.of captions processed in each batch. Defaults to 16.
        part_size (int, optional): No.of embeddings saved per file. Defaults to 10000.
        model_name (str, optional):
            Hugginface pre-trained CLIP identifier. Defaults to "openai/clip-vit-large-patch14".
    """

    with open(caption_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    captions = [entry["caption"] for entry in data]
    os.makedirs(output_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    text_encoder = CLIPTextModel.from_pretrained(model_name).to(device)

    text_embeddings = []
    part_idx = 0
    total_saved = 0

    progress = tqdm(
        range(0, len(captions), batch_size),
        desc=f"Embedding {os.path.basename(caption_json_path)}"
    )
    for i in progress:
        captions_batch = captions[i:i+batch_size]

        try:
            tokens = tokenizer(
                captions_batch,
                padding="max_length",
                truncation=True,
                return_tensors="pt").to(device)

            with torch.no_grad():
                text_features = text_encoder(**tokens).last_hidden_state
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            text_embeddings.append(text_features.cpu())

            if len(text_embeddings) * batch_size >= part_size:
                text_tensor = torch.cat(text_embeddings, dim=0)
                save_path = os.path.join(output_dir, f"text_embeddings_part_{part_idx}.pt")
                torch.save(text_tensor, save_path)
                print(f"Saved {text_tensor.shape} to {save_path}")
                total_saved += text_tensor.shape[0]
                part_idx += 1
                text_embeddings = []

        except Exception as e:
            print(f"Batch at index {i} failed: {e}")
            continue

    if text_embeddings:
        text_tensor = torch.cat(text_embeddings, dim=0)
        save_path = os.path.join(output_dir, f"text_embeddings_part_{part_idx}.pt")
        torch.save(text_tensor, save_path)
        print(f"Saved {text_tensor.shape} to {save_path}")
        total_saved += text_tensor.shape[0]

    print(f"Total embeddings saved: {total_saved}")


In [ ]:
start = time.time()
generate_clip_text_embeddings(
    caption_json_path="coco_train_subset.json",
    output_dir="clip_embeddings_train",
    batch_size=16,
    part_size=80000,
    model_name="openai/clip-vit-large-patch14"
)
stop = time.time()
print(f"Time taken: {stop - start:.2f} seconds")


In [ ]:
start = time.time()
generate_clip_text_embeddings(
    caption_json_path="coco_val_cleaned.json",
    output_dir="clip_embeddings_val",
    batch_size=16,
    part_size=60000,
    model_name="openai/clip-vit-large-patch14"
)
stop = time.time()
print(f"Time taken: {stop - start:.2f} seconds")


# Generating Latents

In [ ]:
import json
import os
from PIL import Image
import torch
from torchvision import transforms
from tqdm import tqdm
from diffusers import AutoencoderKL

def generate_vae_latents(
    caption_json_path,
    image_base_folder,
    output_dir,
    part_size=10000,
    model_path="CompVis/stable-diffusion-v1-4"
):
    """ Generate and save VAE Latents for the images

    This function loads the image paths from the JSON file, preprocess each image, encodes
    them using VAE encoder and save the resulting latents in parts to .pt file inside
    given directory.

    Args:
        caption_json_path (str): Path to JSON file containing image-caption pairs
        image_base_folder (str): Path to the folder where COCO images were saved
        output_dir (str): Path to directory to save the generated latents
        part_size (int, optional): No.of latents saved per file.. Defaults to 10000.
        model_path (str, optional):
            Hugging face model path for the VAE. Defaults to "CompVis/stable-diffusion-v1-4".
    """
    with open(caption_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_paths = [os.path.join(image_base_folder, item["file_name"]) for item in data]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    vae = AutoencoderKL.from_pretrained(model_path, subfolder="vae").to(device).eval()

    preprocess = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ])

    os.makedirs(output_dir, exist_ok=True)

    latents = []
    part_idx = 0
    total_saved = 0

    for i, path in enumerate(tqdm(image_paths, desc="Encoding VAE latents")):
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = preprocess(img).unsqueeze(0).to(device)

            with torch.no_grad():
                latent = vae.encode(img_tensor).latent_dist.sample() * 0.18215

            latents.append(latent.cpu())

            if len(latents) >= part_size:
                latents_tensor = torch.cat(latents)
                torch.save(
                    latents_tensor,
                    os.path.join(output_dir, f"vae_latents_part_{part_idx}.pt"))
                print(f"Saved {latents_tensor.shape} to vae_latents_part_{part_idx}.pt")
                total_saved += latents_tensor.shape[0]
                latents = []
                part_idx += 1

        except Exception as e:
            print(f"Skipping {path}: {e}")
            continue

    if latents:
        latents_tensor = torch.cat(latents)
        torch.save(latents_tensor, os.path.join(output_dir, f"vae_latents_part_{part_idx}.pt"))
        print(f"Saved {latents_tensor.shape} to vae_latents_part_{part_idx}.pt")
        total_saved += latents_tensor.shape[0]

    print(f"Total VAE latents saved: {total_saved}")


In [ ]:
start = time.time()
generate_vae_latents(
    caption_json_path="coco_train_subset.json",
    image_base_folder="coco_dataset/images/train2017",
    output_dir="vae_latents_train",
    part_size=80000
)
stop = time.time()
print(f"Time taken: {stop - start:.2f} seconds")


In [ ]:
start = time.time()
generate_vae_latents(
    caption_json_path="coco_val_cleaned.json",
    image_base_folder="coco_dataset/images/val2017",
    output_dir="vae_latents_val",
    part_size=60000
)
stop = time.time()
print(f"Time taken: {stop - start:.2f} seconds")


# Training Pipeline

In [ ]:
def train_unet_model(
    unet,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    mse_loss,
    noise_scheduler,
    num_epochs=6,
    device="cuda",
    save_dir="epochs",
    csv_log="training_log.csv"
):
    """ Train a UNet to learn predicting the noise

    This function trains a UNet to predict the noise added to the latent during a forward
    diffusion step. The model learns to denoise the latent guided by the CLIP text embedding.
    The noise is added to the latent at each timestep by the noise schedular and UNet learns
    denoising by predicting the noise added.

    Args:
        unet (nn.Module): UNet model to be trained
        train_loader (DataLoader): DataLoader containing training batches
        val_loader (DataLoader): DataLoader containing validation batches
        optimizer (Optimizer): optimizer to update model weights
        scheduler (scheduler): Learning rate schedular to update the learning rate per epoch
        mse_loss (nn.Module): Loss function for training
        noise_scheduler (noise_scheduler): Scheduler controlling noise addition during training
        num_epochs (int, optional): Total no.of training epochs. Defaults to 6.
        device (str, optional): Device to run training on (cuda/cpu). Defaults to "cuda".
        save_dir (str, optional):
            Path to the directory to save model checkpoints. Defaults to "epochs".
        csv_log (str, optional):
            Path to csv file to log train/val loss. Defaults to "training_log.csv".
    """

    os.makedirs(save_dir, exist_ok=True)


    best_val_loss = float("inf")
    train_losses = []
    val_losses = []

    with open(csv_log, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Epoch", "Train Loss", "Validation Loss", "Epoch Time (sec)"])

        for epoch in range(num_epochs):
            start_time = time.time()
            unet.train()
            total_train_loss = 0

            for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
                try:
                    latents = batch["latent"].float().to(device)
                    text_embed = batch["embedding"].float().to(device)
                    noise = torch.randn_like(latents)

                    if torch.rand(1).item() < 0.1:
                        text_embed = torch.zeros_like(text_embed)

                    timesteps = torch.randint(
                        0,
                        noise_scheduler.config.num_train_timesteps,
                        (latents.shape[0],), device=device).long()

                    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                    noise_pred = unet(
                        noisy_latents,
                        timesteps,
                        encoder_hidden_states=text_embed).sample

                    loss = mse_loss(noise_pred, noise)
                    optimizer.zero_grad()
                    loss.backward()
                    clip_grad_norm_(unet.parameters(), max_norm=1.0)
                    optimizer.step()

                    total_train_loss += loss.item()
                    train_losses.append(loss.item())

                    if step % 10 == 0:
                        print(f"[Epoch {epoch+1} | Step {step}] Train Loss: {loss.item():.4f}")

                except RuntimeError as e:
                    print(f"[Step {step}] Runtime error: {e}")
                    torch.cuda.empty_cache()
                    continue

            avg_train_loss = total_train_loss / len(train_loader)
            print(f"Epoch {epoch+1} Avg Train Loss = {avg_train_loss:.4f}")

            unet.eval()
            total_val_loss = 0

            with torch.no_grad():
                for batch in val_loader:
                    latents = batch["latent"].float().to(device)
                    text_embed = batch["embedding"].float().to(device)
                    noise = torch.randn_like(latents)

                    timesteps = torch.randint(
                        0,
                        noise_scheduler.config.num_train_timesteps,
                        (latents.shape[0],), device=device).long()
                    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                    noise_pred = unet(
                        noisy_latents,
                        timesteps,
                        encoder_hidden_states=text_embed).sample

                    val_loss = mse_loss(noise_pred, noise)
                    total_val_loss += val_loss.item()

            avg_val_loss = total_val_loss / len(val_loader)
            val_losses.append(avg_val_loss)
            print(f"Epoch {epoch+1} Avg Validation Loss = {avg_val_loss:.4f}")

            epoch_time = time.time() - start_time

            writer.writerow([epoch+1, avg_train_loss, avg_val_loss, epoch_time])

            torch.save(unet.state_dict(), os.path.join(save_dir, f"unet_epoch_{epoch+1}.pt"))
            print(f"Saved model at epoch {epoch+1}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(unet.state_dict(), os.path.join(save_dir, "unet_best.pt"))
                print(f"Saved best model with val loss {best_val_loss:.4f}")

            scheduler.step()

    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Training Loss")
    plt.plot(
        torch.linspace(
            0,
            len(train_losses),
            steps=len(val_losses)),
            val_losses,
            label="Validation Loss",
            color="orange"
        )
    plt.title("Training and Validation Loss Curve")
    plt.xlabel("Training Steps")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

    print("Training complete.")


In [ ]:
class LDMText2ImageDataset(Dataset):
    """ Dataset class to pair VAE latents and CLIP embeddings

    Loads the latent and embedding .pt files, concat them if there are more than 1
    and make sure both latents and embeddings are of same lenght. Returns a dict
    with latent-embedding pairs.

    Args:
        latents_dir (str): Path to the directory where generated latents are saved
        embeddings_dir (str) : Patht to the direcotory where generated embeddings are saved
    """
    def __init__(self, latents_dir, embeddings_dir):
        self.latent_tensors = self._load_all_parts(latents_dir)
        self.embedding_tensors = self._load_all_parts(embeddings_dir)

        assert len(self.latent_tensors) == len(self.embedding_tensors), (
            f"Mismatch: {len(self.latent_tensors)} latents "
            f"vs {len(self.embedding_tensors)} embeddings"
        )


    def _load_all_parts(self, folder):
        all_parts = sorted(
            [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".pt")]
        )
        all_data = [torch.load(p) for p in all_parts]
        return torch.cat(all_data, dim=0)

    def __len__(self):
        return len(self.latent_tensors)

    def __getitem__(self, idx):
        return {
            "latent": self.latent_tensors[idx],
            "embedding": self.embedding_tensors[idx]
        }


In [ ]:
train_dataset = LDMText2ImageDataset(
    latents_dir="vae_latents_train",
    embeddings_dir="clip_embeddings_train"
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


In [ ]:
val_dataset = LDMText2ImageDataset(
    latents_dir="vae_latents_val",
    embeddings_dir="clip_embeddings_val"
)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet").to(device).train()


In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
noise_scheduler.set_timesteps(1000)

optimizer = AdamW(unet.parameters(), lr=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=5)
mse_loss = nn.MSELoss()


In [ ]:
train_unet_model(
  unet = unet,
  train_loader = train_loader,
  val_loader = val_loader,
  optimizer = optimizer,
  scheduler = scheduler,
  mse_loss = mse_loss,
  noise_scheduler = noise_scheduler,
  num_epochs = 6,
  device = device,
  save_dir = "epochs",
  csv_log = "training.csv")


# Inference Pipeline

In [ ]:
torch.manual_seed(42)


In [ ]:
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet").to(device)


In [ ]:
model_path = "/content/epochs/unet_best.pt"
unet.load_state_dict(torch.load(model_path, map_location=device))
unet.eval()


In [ ]:
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device).eval()


In [ ]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()


In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.set_timesteps(50)


In [ ]:
def generate_image_from_prompt(
        prompt,
        unet,
        vae,
        tokenizer,
        text_encoder,
        scheduler,
        device,
        guidance_scale=7.5,
        save_dir="generated_images",
        filename=None):

    """ Generate and save an image from a text prompt using trained diffusion model

    This function uses a text-to-image generation pipeline with classifier-free guidance. It
    tokenizes and encodes the prompt, performs denoising through the UNet guided by the
    CLIP text embeddings, decodes the resulting latent with a VAE, and saves the output
    image as a PNG.

    Args:
        prompt (str): Text prompt to generate the image from
        unet (nn.Module): UNet model to be trained
        vae (AutoencoderKL): VAE decoder for converting latent to image space.
        tokenizer (CLIPTokenizer): Tokenizer to process the input prompt.
        text_encoder (CLIPTextModel): Model to convert tokens to embeddings.
        scheduler (Scheduler): Noise scheduler
        device (str): Device to run inference on (e.g., "cuda" or "cpu").
        scheduler (scheduler): Learning rate schedular to update the learning rate per epoch
        guidance_scale (float, optional): Strength of classifier-free guidance. Defaults to 7.5.
        save_dir (str, optional):
            Directory to save the generated image. Defaults to "generated_images".
        filename (str, optional): Custom filename for the image. If None, uses prompt-based name.


    Returns:
        Image: Generated Image
    """

    os.makedirs(save_dir, exist_ok=True)

    with torch.no_grad():
        cond_tokens = tokenizer(
            [prompt],
            padding="max_length",
            truncation=True,
            return_tensors="pt").to(device)
        cond_embed = text_encoder(**cond_tokens).last_hidden_state
        cond_embed = cond_embed / cond_embed.norm(dim=-1, keepdim=True)

        uncond_embed = torch.zeros_like(cond_embed).to(device)
        text_embed = torch.cat([uncond_embed, cond_embed], dim=0)

    latent = torch.randn(1, 4, 32, 32).to(device)

    for t in scheduler.timesteps:
        latent_in = latent.repeat(2, 1, 1, 1)
        with torch.no_grad():
            noise_pred = unet(latent_in, t, encoder_hidden_states=text_embed).sample
        noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
        latent = scheduler.step(noise_pred, t, latent).prev_sample

    with torch.no_grad():
        decoded = vae.decode(latent / 0.18215).sample

    image = to_pil_image(decoded.squeeze().clamp(-1, 1) * 0.5 + 0.5)

    if filename is None:
        filename = f"{prompt.replace(' ', '_')[:40]}.png"

    image_path = os.path.join(save_dir, filename)
    image.save(image_path)
    print(f"Saved: {image_path}")

    return image


In [ ]:
prompt = "beautiful mountains with snow"
img = generate_image_from_prompt(
    prompt=prompt,
    unet=unet,
    vae=vae,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    scheduler=scheduler,
    device=device,
    guidance_scale=6,
)

plt.imshow(img)
plt.axis("off")
plt.title(prompt)
plt.show()


# Evaluation

In [ ]:
def generate_multiple_images_prompt_file(
        prompt_file,
        unet,
        vae,
        tokenizer,
        text_encoder,
        scheduler,
        device,
        guidance_scale=7.5,
        save_dir="generated_samples",
        samples=1000
        ):
    """Generate and save muntiple images from a text prompts that are in the prompt file

    Args:
        prompt_file (str):
            Path of the JSON file that contains prompts for which images should be generated
        unet (nn.Module): UNet model to be trained
        vae (AutoencoderKL): VAE decoder for converting latent to image space.
        tokenizer (CLIPTokenizer): Tokenizer to process the input prompt.
        text_encoder (CLIPTextModel): Model to convert tokens to embeddings.
        scheduler (Scheduler): Noise scheduler
        device (str): Device to run inference on (e.g., "cuda" or "cpu").
        scheduler (scheduler): Learning rate schedular to update the learning rate per epoch
        guidance_scale (float, optional): Strength of classifier-free guidance. Defaults to 7.5.
        save_dir (str, optional):
            Directory to save the generated image. Defaults to "generated_samples".
        samples (int, optional): No.of images to be generated from the file. Defaults to 1000.
    """

    os.makedirs(save_dir, exist_ok=True)

    with open(prompt_file, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    sampled_prompts = [entry["caption"] for entry in val_data[:samples]]

    prompt_image_map = []

    for i, prompt in enumerate(tqdm(sampled_prompts, desc="Generating Images")):
        try:
            filename = f"gen_{i:04d}.png"

            _ = generate_image_from_prompt(
                prompt=prompt,
                unet=unet,
                vae=vae,
                tokenizer=tokenizer,
                text_encoder=text_encoder,
                scheduler=scheduler,
                device=device,
                guidance_scale=guidance_scale,
                save_dir=save_dir,
                filename=filename
            )

            prompt_image_map.append({"filename": filename, "caption": prompt})

        except Exception as e:
            print(f"Error with prompt {i}: {e}")
            continue

    with open(os.path.join(save_dir, "captions.json"), "w", encoding="utf-8") as f:
        json.dump(prompt_image_map, f, indent=2)

    print(f"Generation complete. Saved {len(prompt_image_map)} image-caption pairs.")


In [ ]:
generate_multiple_images_prompt_file(prompt_file="/content/coco_val_cleaned.json",
                                     unet=unet,
                                     vae=vae,
                                     tokenizer=tokenizer,
                                     text_encoder=text_encoder,
                                     scheduler=scheduler,
                                     device=device,
                                     guidance_scale=7.5,
                                     save_dir="generated_samples",
                                     samples=1000)


In [ ]:
clip_model, clip_preprocess = clip.load("ViT-L/14", device=device)
clip_model.eval()


In [ ]:
def compute_clip_score(img_path, prompt, model, preprocess, device="cuda"):
    """ Compute CLIP similarity scores betweeen image and text prompt

    This function encodes given image and prompt using CLIP model, computes
    cosine similarity between the embeddings, and returns it as score.

    Args:
        img_path (str): Path to the generated image
        prompt (str): Text prompt used for image
        model (CLIP model): CLIP model to encode image and text
        preprocess (Function): Preprocessing function for images
        device (str, optional):
            Device to run inference pipeline on (cuda/cpu). Defaults to "cuda".

    Returns:
        float: Cosine similarity score between image and text embeddings
    """
    try:
        image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        text = clip.tokenize([prompt]).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)
            text_features = model.encode_text(text)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        score = torch.cosine_similarity(image_features, text_features).item()
        return score
    except Exception as e:
        print(f"Failed on {img_path}: {e}")
        return None


In [ ]:
with open("generated_samples/captions.json", "r", encoding="utf-8") as f:
    data = json.load(f)

clip_scores = []

for item in tqdm(data, desc="CLIP Scoring"):
    prompt = item["caption"]
    path = os.path.join("generated_1k", item["filename"])
    score = compute_clip_score(path, prompt, clip_model, clip_preprocess)
    if score is not None:
        clip_scores.append(score)

print(f"\nAverage CLIP Score: {np.mean(clip_scores):.4f}")
print(f"Min: {np.min(clip_scores):.4f}, Max: {np.max(clip_scores):.4f}")


In [ ]:
plt.hist(clip_scores, bins=20, color="skyblue")
plt.title("CLIP Score Distribution")
plt.xlabel("CLIP Score")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


In [ ]:
shutil.make_archive("generated_samples", "zip", "generated_samples")


In [ ]:
# files.download("generated_samplPes.zip")
